In [1]:
import torch
import numpy as np

def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

seed_everything(42)

In [2]:
import sys
from pathlib import Path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


from IPython import get_ipython
from pprint import pformat
formatter = get_ipython().display_formatter.formatters['text/plain']
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))

<function IPython.lib.pretty._seq_pprinter_factory.<locals>.inner(obj, p, cycle)>

In [3]:
from statistics import mean

from src.editors import EditRequest
from src.evaluate import EvalPair, EvalRecord, MetricResult, EvalSuite, EvalResult


def aggregate_eval_results(results: list[EvalResult]) -> EvalResult:
    if not results:
        return EvalResult()

    def _aggregate_metric(metric_name: str) -> MetricResult | None:
        metric_results = [getattr(result, metric_name) for result in results]
        metric_results = [metric for metric in metric_results if metric is not None]
        if not metric_results:
            return None

        details: list[EvalRecord] = [
            record
            for metric in metric_results
            for record in metric.details
        ]
        score = mean(metric.score for metric in metric_results)

        return MetricResult(score=score, details=details)

    return EvalResult(
        success=_aggregate_metric("success"),
        locality=_aggregate_metric("locality"),
        portability=_aggregate_metric("portability"),
    )


def format_step_log(step: int, total_steps: int, result: EvalResult) -> str:
    parts: list[str] = []

    if result.success is not None:
        parts.append(f"success: {result.success.score:.4f} (n={len(result.success.details)})")
    if result.locality is not None:
        parts.append(f"locality: {result.locality.score:.4f} (n={len(result.locality.details)})")
    if result.portability is not None:
        parts.append(f"portability: {result.portability.score:.4f} (n={len(result.portability.details)})")

    metrics_text = " --- ".join(parts) if parts else "no metrics"
    return f"[{step}/{total_steps}] {metrics_text}"

In [4]:
from dataclasses import asdict
from time import perf_counter
from typing import Literal

from tqdm.notebook import trange

from src.editors import Editor
from src.evaluate import evaluate


async def run_benchmark(
    editor: Editor,
    edit_requests: list[EditRequest],
    eval_suites: list[EvalSuite],
    eval_batch_size: int,
    eval_method: Literal["teacher_forcing", "llm_judge"],
    eval_milestones: list[int],
    **kwargs
) -> list[dict]:

    if not eval_milestones:
        raise ValueError("eval_milestones must not be empty")

    eval_milestones = sorted(eval_milestones)
    if eval_milestones[0] <= 0:
        raise ValueError("Milestones must be positive integers")

    max_step = eval_milestones[-1]
    if len(edit_requests) < max_step or len(eval_suites) < max_step:
        raise ValueError(
            f"Need at least {max_step} requests/suites, got {len(edit_requests)} requests and {len(eval_suites)} suites."
        )

    dataset_name = kwargs.get("dataset_name")
    dataset_variant = kwargs.get("dataset_variant")
    editing_method = kwargs.get("editing_method")
    edited_model = kwargs.get("edited_model")
    print(
        f"[run-start] dataset={dataset_name}/{dataset_variant} editing_method={editing_method} edited_model={edited_model} "
        f"eval_method={eval_method} milestones={eval_milestones} batch_size={eval_batch_size}"
    )

    run_start = perf_counter()

    milestones = set(eval_milestones)
    milestone_results: list[dict] = []

    for step in trange(1, max_step + 1, desc="Benchmark"):
        editor.edit(edit_requests[step - 1])

        if step not in milestones:
            continue

        eval_results = await evaluate(
            editor.model,
            editor.tokenizer,
            eval_suites[:step],
            method=eval_method,
            batch_size=eval_batch_size,
            **kwargs
        )

        step_result = aggregate_eval_results(eval_results)
        print(format_step_log(step, max_step, step_result))

        milestone_results.append({
            "step": step,
            "edits": [
                {
                    "prompt": req.prompt,
                    "old_ground_truth": req.old_ground_truth,
                    "new_ground_truth": req.new_ground_truth,
                }
                for req in edit_requests[:step]
            ],
            "result": asdict(step_result)
        })

    elapsed = perf_counter() - run_start
    print(f"[run-end] dataset={dataset_name}/{dataset_variant} elapsed={elapsed:.2f}s ({elapsed / 60:.2f} min)")
    return milestone_results

In [5]:
# --- Benchmark configuration ---
dataset_variant = ["cloze", "qna"][0]
model_name = ["qwen2.5-7b"][0]
editing_method = ["ft", "rome", "memit", "grace", "wise", "alpha_edit"][5]

eval_batch_size = 16
eval_method = ["teacher_forcing", "llm_judge"][1]
eval_milestones = [1, 5, 10, 50, 100]
n_records = eval_milestones[-1]

model_generation_config = {
    "max_new_tokens": 32,
    # "do_sample": True,
    # "temperature": 0.7,
    # "top_p": 0.8,
    # "top_k": 20,
}

llm_judge_config = {
    "base_url": "http://74.48.90.141:10038/v1",
    "api_key": "local",
    "judge_model": "openai/gpt-oss-20b",
    # "judge_model": "Qwen/Qwen3-32B-AWQ",
    "judge_extra_config": {
        "temperature": 0.1,
        "extra_body": {
            "reasoning_effort": "medium"
        }
        # "presence_penalty": 1.5,
        # "extra_body": {
        #     "chat_template_kwargs": {"enable_thinking": True}
        # }
    },
    "max_concurrency": 4,
    "max_retries": 5,
    # "init_retry_delay": 1.0,
    # "max_retry_delay": 30.0,
}

In [6]:
from datasets import load_from_disk
from src.editors import load_editor

# --- Load editor + datasets ---
editor = load_editor(f"../src/config/{editing_method}/{model_name}.yaml")
cfact = load_from_disk("../data/datasets/cfact")
mquake = load_from_disk("../data/datasets/mquake")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [7]:
import random

from src.evaluate import save_benchmark_results


def select_and_split_pairs(
    pairs: list[tuple[EditRequest, EvalSuite]],
    n_edits: int
) -> tuple[list[EditRequest], list[EvalSuite]]:

    if n_edits <= 0:
        raise ValueError("n must be > 0")
    if n_edits > len(pairs):
        raise ValueError(f"Requested {n_edits} edits, but only {len(pairs)} edits are available.")

    shuffled = pairs.copy()
    random.shuffle(shuffled)
    selected = shuffled[:n_edits]

    edit_requests = [request for request, _ in selected]
    eval_suites = [suite for _, suite in selected]
    return edit_requests, eval_suites

In [8]:
# --- Run cfact benchmark (success + locality) ---
def build_cfact_pairs(dataset) -> list[tuple[EditRequest, EvalSuite]]:
    def build_pair(record):
        edit_request = EditRequest(
            record['subject'],
            record['prompt'],
            record['old_ans'],
            record['new_ans']
        )
        eval_suite = EvalSuite(
            success=[EvalPair(record['prompt'], record['new_ans'])] +\
                    [EvalPair(prompt, record['new_ans'])
                     for prompt in record['rephrases']],
            locality=[EvalPair(pair['prompt'], pair['ans'])
                      for pair in record['locality']]
        )
        return edit_request, eval_suite
    return [build_pair(record) for record in dataset]


cfact_pairs = build_cfact_pairs(cfact[dataset_variant])
cfact_requests, cfact_suites = select_and_split_pairs(cfact_pairs, n_records)

editor.reset()

cfact_milestone_results = await run_benchmark(
    editor=editor,
    edit_requests=cfact_requests,
    eval_suites=cfact_suites,
    eval_batch_size=eval_batch_size,
    eval_method=eval_method,
    eval_milestones=eval_milestones,
    dataset_name="cfact",
    dataset_variant=dataset_variant,
    editing_method=editing_method,
    edited_model=model_name,
    model_generation_config=model_generation_config,
    llm_judge_config=llm_judge_config,
)

save_benchmark_results(
    results=cfact_milestone_results,
    eval_method=eval_method,
    dataset_variant=dataset_variant,
    edited_model=model_name,
    editing_method=editing_method,
    milestones=eval_milestones,
    dataset_name="cfact",
)

[run-start] dataset=cfact/cloze editing_method=alpha_edit edited_model=qwen2.5-7b eval_method=llm_judge milestones=[1, 5, 10, 50, 100] batch_size=16


Benchmark:   0%|          | 0/100 [00:00<?, ?it/s]

LLM Judge Evaluation:   0%|          | 0/14 [00:00<?, ?record/s]

Evaluating 14 records. 5 retries left.
[1/100] success: 0.7500 (n=4) --- locality: 0.0000 (n=10)


LLM Judge Evaluation:   0%|          | 0/70 [00:00<?, ?record/s]

...

LLM Judge Evaluation:   0%|          | 0/700 [00:00<?, ?record/s]

Evaluating 700 records. 5 retries left.
[50/100] success: 0.7400 (n=200) --- locality: 0.3060 (n=500)


LLM Judge Evaluation:   0%|          | 0/1400 [00:00<?, ?record/s]

Evaluating 1400 records. 5 retries left.
[100/100] success: 0.7625 (n=400) --- locality: 0.3080 (n=1000)
[run-end] dataset=cfact/cloze elapsed=1423.87s (23.73 min)


PosixPath('/workspace/mnemo/data/evaluation/llm_judge/cloze/qwen2.5-7b/alpha_edit/1-5-10-50-100/cfact.json')

In [9]:
# --- Run mquake benchmark (portability) ---
def build_mquake_pairs(dataset) -> list[tuple[EditRequest, EvalSuite]]:
    def build_pair(record):
        edit_request = EditRequest(
            record['subject'],
            record['prompt'],
            record['old_ans'],
            record['new_ans']
        )
        eval_suite = EvalSuite(
            portability=[EvalPair(prompt, record['portability']['new_ans'])
                         for prompt in record['portability']['prompts']]
        )
        return edit_request, eval_suite
    return [build_pair(record) for record in dataset]


mquake_pairs = build_mquake_pairs(mquake[dataset_variant])
mquake_requests, mquake_suites = select_and_split_pairs(mquake_pairs, n_records)

editor.reset()

mquake_milestone_results = await run_benchmark(
    editor=editor,
    edit_requests=mquake_requests,
    eval_suites=mquake_suites,
    eval_batch_size=eval_batch_size,
    eval_method=eval_method,
    eval_milestones=eval_milestones,
    dataset_name="mquake",
    dataset_variant=dataset_variant,
    editing_method=editing_method,
    edited_model=model_name,
    model_generation_config=model_generation_config,
    llm_judge_config=llm_judge_config,
)

save_benchmark_results(
    results=mquake_milestone_results,
    eval_method=eval_method,
    dataset_variant=dataset_variant,
    edited_model=model_name,
    editing_method=editing_method,
    milestones=eval_milestones,
    dataset_name="mquake",
)

[run-start] dataset=mquake/cloze editing_method=alpha_edit edited_model=qwen2.5-7b eval_method=llm_judge milestones=[1, 5, 10, 50, 100] batch_size=16


Benchmark:   0%|          | 0/100 [00:00<?, ?it/s]

LLM Judge Evaluation:   0%|          | 0/3 [00:00<?, ?record/s]

Evaluating 3 records. 5 retries left.
[1/100] portability: 0.0000 (n=3)


LLM Judge Evaluation:   0%|          | 0/15 [00:00<?, ?record/s]

...

LLM Judge Evaluation:   0%|          | 0/150 [00:00<?, ?record/s]

Evaluating 150 records. 5 retries left.
[50/100] portability: 0.1000 (n=150)


LLM Judge Evaluation:   0%|          | 0/300 [00:00<?, ?record/s]

Evaluating 300 records. 5 retries left.
[100/100] portability: 0.1433 (n=300)
[run-end] dataset=mquake/cloze elapsed=1047.07s (17.45 min)


PosixPath('/workspace/mnemo/data/evaluation/llm_judge/cloze/qwen2.5-7b/alpha_edit/1-5-10-50-100/mquake.json')